<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/05-skills-and-progressive-disclosure.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Skills 与渐进式披露（概念）

**目标：** 理解如何给 agent 提供深入、可复用的知识，*而不用*在每一轮为它支付上下文代价（**渐进式披露**模式），并理解它的具体实例 **Agent Skill**（`SKILL.md`），足以让你会去用它、也能为这个选择辩护。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

> **这是一个概念 notebook，没有代码要跑。** 和 [MCP](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/04-mcp-and-the-tool-ecosystem.zh.ipynb) 一样，Skills 是一个存在于 agent 宿主内部的*打包 + 运行时*层（Claude Code、Claude 应用、Agent SDK、Messages API 的代码执行容器），而不是你能对 Groq 上一次原生 `chat.completions` 调用去调用的东西。所以我们讲的是**经久耐用的模式**，并展示真实 `SKILL.md` 的形状作为参考，和我们讲 MCP 的方式一样。你已经在 `01`–`03` 里亲手搭过工具循环；这是生态故事的另一半。

## 问题：上下文是一份预算

到现在，你已经从上下文窗口的两侧都感受到了挤压：

- 在 [`01-model-apis/04-context-and-caching`](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/01-model-apis/04-context-and-caching.zh.ipynb) 里你看到 prompt 里的每个 token 都要花钱、都要花延迟，*每一次调用*都是。
- 在 [`02-tool-design`](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/02-tool-design.zh.ipynb) 里你看到你交给模型的每个工具 schema 都永久占着上下文，太多会撑爆 prompt 并搞乱工具选择。

现在想象真实的工作。你希望你的 agent 在十几个专门任务上真正出色：填一张特定的 PDF 表单、生成一份带品牌的 PowerPoint、遵循公司的故障处理 runbook、按规范格式化一份财务报告。每一项都需要*好几段*指令，可能有示例文件，可能有已验证的脚本。

天真的做法是把这一切全塞进 system prompt。这在算术上就失败了：十几个详细 playbook 就是模型要携带、并且要为之付费的几万 token，而且是在**每一轮**，即使当前任务一个都不沾。而一个试图成为所有事情专家的 prompt，实践中总是可靠地一事无成。

> **🚩 常见错误 ——** 把 system prompt 当成「agent 可能永远需要知道的一切」的无限存储。上下文是你每次调用都要花掉的预算；agent *偶尔*才需要的流程知识不该被*持续*计费。

所以本 notebook 要回答的真正问题是：**如何在只为一个 agent 当下正在使用的那一小片内容支付上下文的同时，给它宽广而深入的能力？**

## 经久耐用的模式：渐进式披露

答案是一个值得带到任何 agent 技术栈的想法，而不只是某一家厂商的：**把*是什么*和*何时/如何*分开，按需加载细节。**

与其用一个巨大的、永远常驻的 prompt，你把每项能力打包成一个自包含的单元，带一个便宜的**标签**（名字 + 一句描述*它做什么、什么时候用*）和一个昂贵的**正文**（完整指令、示例、文件、脚本）。只有标签常驻上下文。模型*只在某个请求匹配到它的描述时*才去读一个单元的正文，而且只有当正文真的要求时，才去读随附文件或运行随附脚本。

这就是**渐进式披露**。它把人们通常混为一谈的两件事解耦：

- agent 宣称的**能力广度**（可以巨大），与
- 携带它的**上下文代价**（在使用前保持在接近零）。

下面的具体实例 Agent Skills 用三个加载级别让这个机制变得生动：

| 级别 | 何时加载 | 大致 token 代价 | 内容 |
|---|---|---|---|
| **1 — 元数据** | 始终（启动时） | **每个 skill 约 100 token** | 只有 `name` + `description` |
| **2 — 指令** | 当某个请求匹配到描述时 | 不到约 5k token | `SKILL.md` 正文 |
| **3 — 资源** | 按需，若正文引用了它们 | 访问前为零 | 随附文件（读入上下文）和脚本（**通过 bash 运行，所以只有其*输出*进入上下文，代码永不进入**） |

两个后果让这不只是一招整洁的 prompt 技巧：

- **你几乎不花代价就能挂上几十项能力。** 每个约 100 token，在某个 skill 真的被触发之前，一整个 skill 库只是个舍入误差。广度不再与上下文预算打架。
- **随附代码换来确定性。** 一个 skill 可以随附一个*已验证*的脚本让 agent 运行，而不是每次都在上下文里重新推导脆弱的逻辑，而且因为只有脚本的输出返回，一个 400 行的工具函数代价为零上下文。这是 `02` 里工具设计的直觉再往前一步：不要让模型去*做*一个正确函数能做的事。

> **⭐ 关键结论 ——** 渐进式披露是那个经久耐用的想法；`SKILL.md` 只是它今天的具体写法。如果你内化了「廉价地宣传、按需加载、把确定性推进随附代码」，那么不管某个 agent 平台管它叫什么，你都能认出这个模式。

## 具体实例：一个 Agent Skill

**Agent Skill** 就是那个模式当下被广泛使用的打包形式。最简单的形式就是*一个包含 `SKILL.md` 文件的文件夹*：YAML frontmatter（那个便宜的标签）加一个 Markdown 正文（指令），旁边可选地放随附脚本、参考资料和模板：

```
pdf-processing/
├── SKILL.md          # required: frontmatter (label) + Markdown body (instructions)
├── scripts/          # optional: code the agent runs via bash (output returns, not the code)
├── references/       # optional: docs the agent reads only when the body points to them
└── assets/           # optional: templates, schemas, example files
```

一个最小、可移植的 `SKILL.md` 长这样（**仅供参考；这里没有东西要跑**）：

```markdown
---
name: pdf-processing
description: Extract text and tables from PDFs, fill forms, and merge documents.
  Use when the user mentions PDFs, forms, or document extraction.
---

# PDF Processing

## Instructions
1. To extract text, open the PDF with `pdfplumber` and read `page.extract_text()`.
2. For form filling, follow the steps in [FORMS.md](references/FORMS.md).
3. To merge documents, run `scripts/merge.py <in1> <in2> <out>`.

## Examples
- "Pull the text out of this contract and summarize it."
- "Fill in this application form with the values I gave you."
```

整个设计都围绕**两个必填字段**，而第二个是人们最常低估的：

- **`name`** —— 一个简短的标识符（小写、连字符）。
- **`description`** —— 模型用来匹配请求、决定是否触发这个 skill 的字符串。它必须**同时说明这个 skill 做什么*以及*什么时候用。** 一个含糊的描述是一个 skill 悄悄永不触发的第一大原因。这和 [`02-tool-design`](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/05-agents/02-tool-design.zh.ipynb) 里「为模型写 schema，而不是为你自己写」是同一条纪律，只是用在标签上而不是工具上。

可移植的开放规范还允许几个可选字段（`license`、`compatibility`、`metadata`，以及 `allowed-tools`，即一个 skill 无需逐次调用审批就能使用的工具）。这里刻意**没有 `version` 字段**：版本管理放在文件之外（git，或平台的版本 API）。个别 agent 宿主（例如 Claude Code）会加自己额外的 frontmatter 键，但那些是宿主专属扩展，不属于可移植内核。

> **⚠️ 生产现实 ——** 同一个 `SKILL.md` *不会*神奇地在各个界面之间共享。你加到某个聊天应用里的 skill、通过 API 注册的 skill，以及躺在某个仓库 `.claude/skills/` 里的 skill，是不同位置的三次独立安装。「写一次」指的是*格式*，不是自动同步。在你承诺某个 skill 可用之前，先搞清楚它实际住在哪里。

## 补全 agent 故事的三件套

你现在已经认识了三个互补的层。一个清晰的心智模型，也是一个很好的面试答案，是它们回答的是不同的问题：

| 层 | 回答 | 它是什么 | 上下文代价 |
|---|---|---|---|
| **Tools**（`01`–`03`、`02-tool-design`） | *agent 能**做**什么？* | 模型能调用的单个函数 + schema | schema **始终**在上下文里 |
| **MCP**（`04-mcp`） | *agent 能**触达**哪些外部系统？* | 一个把 agent 连到 server、数据、API 的开放协议 | 每个接入的工具 schema 都在上下文里 |
| **Skills**（本 notebook） | *agent 应该如何**把一件任务做好**？* | 一个打包好的、渐进式披露的流程 + 资源文件夹 | 触发前约 **100 token** |

它们是组合关系，不是竞争关系。一个有用的记法是：**MCP 把 agent 连接到外部系统；Skills 打包把系统用好所需的流程性知识。** 一个 skill 的指令甚至可以编排由 MCP 提供的工具，而一个 skill 可以随附自己的确定性代码，作为工具运行。Tools 是动词，MCP 是通往外部世界的接线，Skills 是剧本。

> **🔵 信号 ——** 一个能把这三位放到这张网格上、并说清*为什么* Skills 由元数据门控而工具 schema 始终常驻的工程师，展示的是他理解上下文预算是架构约束，而不只是一条账单。这个框架在面试里很吃得开。

## FDE 什么时候会去用一个 Skill（以及什么时候不用）

**这些情况去用 skill：**
- agent 需要**可重复的流程性知识**（一个 runbook、一种公司内部格式、一条多步工作流），它太详细、放不进 system prompt，但只是*偶尔*需要。
- 你想**随附一个已验证的脚本或参考文件**，让 agent 运行正确的代码/读取精确的规范，而不是每次即兴发挥。
- 你在构建**很多能力**，而那个常驻 prompt 变得昂贵或混乱。渐进式披露正是那个泄压阀。
- 你希望这项能力**可移植到多个 agent 宿主**。这个格式是一个被广泛采用、远超其起源的开放标准（见下）。

**这些情况跳过它（保留一个普通 prompt 或工具）：**
- 那条指令**很短且始终相关**。如果它反正每一轮都该在，那么一行 system prompt 比一个 skill 更简单。
- 你需要的是**一次函数调用**，不是一套流程。那是一个工具（`02`），不是 skill。
- 你在**学习这个循环**，这也是为什么本仓库先教你手搭工具。在你了解某个打包层打包的是什么之前，你没法判断它有没有帮助。

> **💡 为什么起源故事重要 ——** Agent Skill 格式起源于 Anthropic，但作为**开放标准**发布（agentskills.io），如今已被多个 agent 产品消费（OpenAI 的 Codex、Google 的 Gemini CLI、各种编辑器助手等等）。这就是你在学一个*经久耐用的模式*而不是某家厂商功能的标志，也是本仓库宁可教原生 API 和具名模式、也不教任何一个封装的原因。

判断仍是本仓库惯常的那条：**先把原生模式搞懂（上下文是预算；廉价地宣传、按需加载），当能力的广度——而不是 demo——足以证明它值得时，再采用这个标准。**

## 安全提示：随附的指令和代码是一处信任面

结束 MCP notebook 的那条警告在这里同样适用，而且更尖锐。一个 skill 是*模型将要遵循的指令*，往往还加上*agent 将要执行的代码*，所以一个装上的 skill 和写它的人一样可信。一个从公开目录拉下来的 skill，其正文可能带间接 prompt 注入，或者带一个做了超出它声称之事的脚本。「它只是个 skill」不是一条安全论证。

像 [`07-security`](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/07-security/01-prompt-injection-and-trust.zh.ipynb) 说的那样对待任何不可信输入一样对待第三方 skill：安装前先读、严格限定 `allowed-tools`、优先沙箱化执行，并对重要动作加人工门禁。渐进式披露让能力*添加*起来很便宜，这让「你添加什么」的纪律变得更重要，而不是更不重要。

**延伸阅读：** [Agent Skills 开放标准](https://agentskills.io)（规范 + 采用它的客户端列表）和 Anthropic 的 [Agent Skills 工程文章](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)。

## 实践与反模式

| ✅ 该做 | ❌ 反模式 |
|---|---|
| 把始终相关、短小的指令留在 prompt 里；把*偶尔用到、详细的*知识打包成 skill | 把每个 playbook 都塞进 system prompt，每一轮为全部付费 |
| 为模型写 `description`：它做什么**以及什么时候用** | 一个含糊的 `description`，然后纳闷为什么这个 skill 从不触发 |
| 把脆弱的逻辑推进一个随附的、已验证的脚本（只有它的输出花上下文） | 让模型每次都在上下文里重新推导同样脆弱的过程 |
| 把装上的 skill 当作可信代码；审阅第三方的（见 `07`） | 装一个公开 skill，就假定它的正文/脚本是安全的 |
| 当*广度*对上下文预算造成压力时才去用 skill | 在你理解它所依附的工具循环之前就去用这个打包层 |

（贯穿整个技术栈的完整清单见 [docs/best-practices-and-anti-patterns.md](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/docs/best-practices-and-anti-patterns.md)。）

## 练习（不写代码，只做推理）

1. **做一遍算术。** 假设你有 12 个详细 playbook，每个约 2k token，一个典型任务会用到其中一个。比较两种做法的每轮上下文代价：(a) 全部 12 个放进 system prompt，(b) 12 个 skill，各约 100 token 元数据，加上那一个被触发的正文。在每天 10 万次调用下，这个比值对成本和延迟意味着什么？
2. **写一个能触发的描述。** 从你自己的工作中挑一个任务，草拟一个 skill 的 `description`（最多两句），把*它做什么、什么时候用*说得足够好，好到模型会因正确的请求触发它、否则忽略它。再写一个坏的版本，说明它为什么会误触发。
3. **把它放到网格上。** 从你的结业项目（第 10 节）里挑一项能力，判断：它是一个 Tool、一条 MCP 连接，还是一个 Skill，或者一个*使用*工具的 Skill？用两句话向面试官说明理由。
4. **脚本 vs prompt。** 说出你项目里的一段逻辑，它目前是「让模型来做」，而如果做成 skill 运行的随附脚本会更可靠。把它移过去，在确定性和上下文代价两方面都买到了什么？
5. **给一个公开 skill 做威胁建模。** 你装了一个第三方的「invoice-formatter」skill，它随附一个脚本。列出它可能伤害你的两种方式，以及每种方式的缓解措施。（学完 [`07-security`](https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/07-security/01-prompt-injection-and-trust.zh.ipynb) 后再回来看看。）